# 전처리 코드
- 전체 데이터셋 사용
- 단일 객체 이미지만 사용
- 클래스당 1,000장 균등 샘플링
- 크롭 없이 원본 이미지 224x224 리사이즈
- bert_text: f'선택된 이미지 속의 폐기물은 {재질}이며, {상태} 특징을 보입니다.' 형식

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

print('라이브러리 로드 완료')

## 0. 경로 설정

In [ ]:
LABEL_DIR         = r'C:\Users\A\Desktop\307.생활폐기물 데이터 활용ㆍ환류\01-1.정식개방데이터\Training\02.라벨링데이터'
IMAGE_DIR         = r'C:\Users\A\Desktop\307.생활폐기물 데이터 활용ㆍ환류\01-1.정식개방데이터\Training\01.원천데이터'
OUTPUT_DIR        = 'output_23cls'
SAMPLES_PER_CLASS = 1000
SEED              = 42
SIZE              = 224

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
IMG_SAVE_DIR = os.path.join(OUTPUT_DIR, 'images')
Path(IMG_SAVE_DIR).mkdir(parents=True, exist_ok=True)

print(f'출력 경로: {OUTPUT_DIR}')
print(f'클래스당 샘플: {SAMPLES_PER_CLASS}장')
print(f'예상 총 샘플: {SAMPLES_PER_CLASS * 23}장')

## 1. 클래스 매핑 정의 (23개)

In [ ]:
# 29개 원본 class_name → 23개 클래스 매핑
CLASS_TO_GROUP = {
    # 재활용 (이물질/다중포장재 없음)
    'c_1': 0,                # 종이
    'c_2_01': 1,             # 종이팩
    'c_2_02': 2,             # 종이컵
    'c_3': 3,                # 캔
    'c_4_01_02': 4,          # 재사용유리
    'c_4_02_01_02': 5,       # 유리 (갈색)
    'c_4_02_02_02': 5,       # 유리 (녹색)
    'c_4_02_03_02': 5,       # 유리 (백색)
    'c_4_03': 5,             # 유리 (기타)
    'c_5_02': 6,             # 페트
    'c_6': 7,                # 플라스틱
    'c_7': 8,                # 비닐
    # 일반쓰레기 (이물질 있음)
    'c_1_01': 9,             # 종이+이물질
    'c_2_02_01': 10,         # 종이컵+이물질
    'c_3_01': 11,            # 캔+이물질
    'c_4_03_01': 12,         # 유리+이물질
    'c_5_01_01': 13,         # 페트+이물질+다중포장재
    'c_5_02_01': 14,         # 페트+이물질
    'c_6_01': 15,            # 플라스틱+이물질
    'c_7_01': 16,            # 비닐+이물질
    # 다중포장재 (분리 후 배출)
    'c_4_01_01': 17,      # 재사용유리+다중포장재
    'c_4_02_01_01': 18,      # 유리+다중포장재
    'c_4_02_02_01': 18,
    'c_4_02_03_01': 18,
    'c_5_01': 19,      # 페트+다중포장재
    # 별도배출
    'c_8_01': 20,      # 스티로폼
    'c_8_02': 20,
    'c_8_01_01': 21,      # 스티로폼+이물질
    'c_9': 22,      # 건전지
}

GROUP_NAMES = {
    0:  '종이',
    1:  '종이팩',
    2:  '종이컵',
    3:  '캔',
    4:  '재사용유리',
    5:  '유리',
    6:  '페트',
    7:  '플라스틱',
    8:  '비닐',
    9:  '종이+이물질',
    10: '종이컵+이물질',
    11: '캔+이물질',
    12: '유리+이물질',
    13: '페트+이물질+다중포장재',
    14: '페트+이물질',
    15: '플라스틱+이물질',
    16: '비닐+이물질',
    17: '재사용유리+다중포장재',
    18: '유리+다중포장재',
    19: '페트+다중포장재',
    20: '스티로폼',
    21: '스티로폼+이물질',
    22: '건전지',
}

# bert_text: f'선택된 이미지 속의 폐기물은 {재질}이며, {상태} 특징을 보입니다.' 형식
CLASS_BERT_TEXT = {
    'c_1': '선택된 이미지 속의 폐기물은 종이이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_2_01': '선택된 이미지 속의 폐기물은 종이팩이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_2_02': '선택된 이미지 속의 폐기물은 종이컵이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_3': '선택된 이미지 속의 폐기물은 금속 캔이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_4_01_02': '선택된 이미지 속의 폐기물은 재사용 유리병이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_4_02_01_02': '선택된 이미지 속의 폐기물은 유리병이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_4_02_02_02': '선택된 이미지 속의 폐기물은 유리병이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_4_02_03_02': '선택된 이미지 속의 폐기물은 유리병이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_4_03': '선택된 이미지 속의 폐기물은 유리류이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_5_02': '선택된 이미지 속의 폐기물은 페트병이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_6': '선택된 이미지 속의 폐기물은 플라스틱 용기이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_7': '선택된 이미지 속의 폐기물은 비닐류이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_1_01': '선택된 이미지 속의 폐기물은 종이이며, 이물질이 묻어 있어 일반쓰레기로 배출해야 하는 특징을 보입니다.',
    'c_2_02_01': '선택된 이미지 속의 폐기물은 종이컵이며, 이물질이 남아 있어 일반쓰레기로 배출해야 하는 특징을 보입니다.',
    'c_3_01': '선택된 이미지 속의 폐기물은 금속 캔이며, 이물질이 남아 있어 일반쓰레기로 배출해야 하는 특징을 보입니다.',
    'c_4_03_01': '선택된 이미지 속의 폐기물은 유리류이며, 이물질이 붙어 있어 일반쓰레기로 배출해야 하는 특징을 보입니다.',
    'c_5_01_01': '선택된 이미지 속의 폐기물은 페트병이며, 이물질이 있고 다중포장재가 결합되어 있어 분리 후 배출해야 하는 특징을 보입니다.',
    'c_5_02_01': '선택된 이미지 속의 폐기물은 페트병이며, 이물질이 남아 있어 일반쓰레기로 배출해야 하는 특징을 보입니다.',
    'c_6_01': '선택된 이미지 속의 폐기물은 플라스틱 용기이며, 이물질이 묻어 있어 일반쓰레기로 배출해야 하는 특징을 보입니다.',
    'c_7_01': '선택된 이미지 속의 폐기물은 비닐류이며, 이물질이 묻어 있어 일반쓰레기로 배출해야 하는 특징을 보입니다.',
    'c_4_01_01': '선택된 이미지 속의 폐기물은 재사용 유리병이며, 다중포장재가 결합되어 있어 분리 후 배출해야 하는 특징을 보입니다.',
    'c_4_02_01_01': '선택된 이미지 속의 폐기물은 유리병이며, 다중포장재가 결합되어 있어 분리 후 배출해야 하는 특징을 보입니다.',
    'c_4_02_02_01': '선택된 이미지 속의 폐기물은 유리병이며, 다중포장재가 결합되어 있어 분리 후 배출해야 하는 특징을 보입니다.',
    'c_4_02_03_01': '선택된 이미지 속의 폐기물은 유리병이며, 다중포장재가 결합되어 있어 분리 후 배출해야 하는 특징을 보입니다.',
    'c_5_01': '선택된 이미지 속의 폐기물은 페트병이며, 다중포장재가 결합되어 있어 분리 후 배출해야 하는 특징을 보입니다.',
    'c_8_01': '선택된 이미지 속의 폐기물은 흰색 스티로폼이며, 이물질이 없는 깨끗한 상태의 특징을 보입니다.',
    'c_8_02': '선택된 이미지 속의 폐기물은 컬러 스티로폼이며, 별도 배출이 필요한 특징을 보입니다.',
    'c_8_01_01': '선택된 이미지 속의 폐기물은 스티로폼이며, 이물질이 묻어 있어 일반쓰레기로 배출해야 하는 특징을 보입니다.',
    'c_9': '선택된 이미지 속의 폐기물은 건전지이며, 별도 수거함에 배출해야 하는 특징을 보입니다.',
}

print('클래스 매핑 완료')
print(f'총 클래스 수: {len(GROUP_NAMES)}')

## STEP 1: JSON 스캔 -> 단일 객체 이미지만 CSV 생성

In [ ]:
label_dir = Path(LABEL_DIR)
json_files = list(label_dir.rglob('*.json'))
print(f'전체 JSON 파일 수: {len(json_files)}')

rows = []
error_count = 0

for json_path in tqdm(json_files, desc='JSON 스캔'):
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        image_filename = data.get('Image', '')
        place = data['Info'].get('PLACE', '실내')
        day_night = data['Info'].get('DAY/NIGHT', '주간')

        objects = data.get('objects', [])

        # 유효한 클래스 객체만 필터링
        valid_objs = [
            obj for obj in objects
            if CLASS_TO_GROUP.get(obj.get('class_name', ''), -1) != -1
        ]
        # 단일 객체 이미지만 사용
        if len(valid_objs) != 1:
            continue

        obj = valid_objs[0]
        class_name = obj['class_name']
        group_id = CLASS_TO_GROUP[class_name]

        rows.append({
            'image_filename': image_filename,
            'class_name': class_name,
            'group_id':  group_id,
            'group_name': GROUP_NAMES[group_id],
            'place': place,
            'day_night': day_night,
            'bert_text': CLASS_BERT_TEXT.get(class_name, '알 수 없음'),
        })

    except Exception as e:
        error_count += 1

df_full = pd.DataFrame(rows)
print(f'\n단일 객체 이미지 수: {len(df_full)} | 오류: {error_count}')
print('\n클래스별 수량:')
print(df_full.groupby(['group_id', 'group_name']).size().sort_index())

## STEP 2: 클래스당 1,000장 균등 샘플링

In [ ]:
sampled_dfs = []

print(f'클래스당 {SAMPLES_PER_CLASS}장 균등 샘플링:\n')
for group_id in range(23):
    group_df = df_full[df_full['group_id'] == group_id]
    available = len(group_df)
    if available == 0:
        print(f'  {GROUP_NAMES[group_id]}: 데이터 없음 - 스킵')
        continue
    n_sample = min(SAMPLES_PER_CLASS, available)
    sampled = group_df.sample(n=n_sample, random_state=SEED)
    sampled_dfs.append(sampled)
    print(f'  {GROUP_NAMES[group_id]}: {n_sample}장 (전체 {available}장 중)')

df_sampled = pd.concat(sampled_dfs).reset_index(drop=True)
print(f'\n총 샘플링: {len(df_sampled)}장')

## STEP 2-2: 이미지 딕셔너리 생성 (파일명 → 실제 경로)

In [ ]:
# 샘플링된 파일명만 딕셔너리 생성
target_files = set(df_sampled['image_filename'].tolist())

img_dict = {}
for img_path in Path(IMAGE_DIR).rglob('*.jpg'):
    if img_path.name in target_files:
        img_dict[img_path.name] = str(img_path)

print(f'찾은 이미지 수: {len(img_dict)} / {len(target_files)}')

## STEP 3: 이미지 224x224 리사이즈 저장

In [ ]:
img_paths   = []
error_count = 0

for i, row in tqdm(df_sampled.iterrows(), total=len(df_sampled), desc='리사이즈'):
    try:
        img_path = img_dict.get(row['image_filename'], '')
        if not img_path:
            raise FileNotFoundError(f"파일 없음: {row['image_filename']}")

        img = Image.open(img_path).convert('RGB')
        resized = img.resize((SIZE, SIZE), Image.BILINEAR)

        save_name = f'img_{i:07d}_g{row["group_id"]}_{row["class_name"]}.jpg'
        save_path = os.path.join(IMG_SAVE_DIR, save_name)
        resized.save(save_path, 'JPEG', quality=90)
        img_paths.append(save_path)

    except Exception as e:
        img_paths.append('')
        error_count += 1

df_sampled = df_sampled.copy()
df_sampled['img_path'] = img_paths
df_sampled = df_sampled[df_sampled['img_path'] != ''].reset_index(drop=True)

sample_csv = os.path.join(OUTPUT_DIR, 'sampled_dataset.csv')
df_sampled.to_csv(sample_csv, index=False, encoding='utf-8-sig')

print(f'\n리사이즈 완료: {len(df_sampled)}장 | 오류: {error_count}')
print('\n클래스별 결과:')
print(df_sampled.groupby(['group_id', 'group_name']).size().sort_index())

## STEP 3-1: 결과 시각화 확인

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 6, figsize=(20, 14))
for idx, ax in enumerate(axes.flat):
    if idx >= 23:
        ax.axis('off')
        continue
    group_df = df_sampled[df_sampled['group_id'] == idx]
    if len(group_df) == 0:
        ax.axis('off')
        continue
    row = group_df.iloc[0]
    img = Image.open(row['img_path'])
    ax.imshow(img)
    ax.set_title(f"{row['group_name']}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## STEP 4: Train / Val / Test 분할 (75/15/10)

In [ ]:
train_df, temp_df = train_test_split(
    df_sampled, test_size=0.25,
    stratify=df_sampled['group_id'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.4,
    stratify=temp_df['group_id'], random_state=SEED
)

train_df.to_csv(os.path.join(OUTPUT_DIR, 'train.csv'), index=False, encoding='utf-8-sig')
val_df.to_csv(os.path.join(OUTPUT_DIR,   'val.csv'),   index=False, encoding='utf-8-sig')
test_df.to_csv(os.path.join(OUTPUT_DIR,  'test.csv'),  index=False, encoding='utf-8-sig')

print('데이터 분할 완료:')
print(f'  Train: {len(train_df)}장')
print(f'  Val:   {len(val_df)}장')
print(f'  Test:  {len(test_df)}장')
print('\n전처리 완료!')
print(f'출력 폴더: {OUTPUT_DIR}')